In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Load the Dataset

In [3]:
df = pd.read_csv("data/taxi_pickups_area.csv")
areas = [c for c in df.columns.to_list() if c != 'Trip Start Timestamp']
df['Trip Start Timestamp'] = pd.to_datetime(df['Trip Start Timestamp'])
df.shape

(8064, 79)

Train validation split

In [4]:
train_size = int(len(df) * 0.8)

train = df[:train_size]
val = df[train_size:]

train.shape

(6451, 79)

In [4]:
val.shape

(1613, 79)

Time Series Preprocessing

Check Missing values and time frequency

In [5]:
# missing = train[train.isnan()].count()
train[train.isna()].count()

Trip Start Timestamp        0
Pickup Community Area_0     0
Pickup Community Area_1     0
Pickup Community Area_2     0
Pickup Community Area_3     0
                           ..
Pickup Community Area_73    0
Pickup Community Area_74    0
Pickup Community Area_75    0
Pickup Community Area_76    0
Pickup Community Area_77    0
Length: 79, dtype: int64

no missing values detected

Check time Frequency

In [6]:
pd.infer_freq(train[train.columns[0]])

'15min'

Time Frequency constant 15-min interval

Handle anomalies with STL Decomposition for each area

In [5]:
from statsmodels.tsa.seasonal import STL

# def detect_anomalies(serie, robust=True, period=672, threshold=3.0): # an be used!

#     stl = STL(serie, robust=robust, period=period)
#     result = stl.fit()
#     resid = result.resid

#     # Robust center/scale — mean/std get skewed by the very anomalies we want to find
#     median = resid.median()
#     mad = np.median(np.abs(resid - median))
#     mad_std = mad * 1.4826 if mad != 0 else resid.std()  # fallback if MAD is 0

#     lower = median - (threshold * mad_std)
#     upper = median + (threshold * mad_std)

#     return serie[(resid < lower) | (resid > upper)]


def detect_anomalies(serie, robust=True, period=672):

    stl = STL(serie, robust=robust, period=period)

    result = stl.fit()

    resid =  result.resid

    resid_m = resid.mean()
    resid_dv = resid.std()

    lower = resid_m - (3 * resid_dv) 
    upper = resid_m + (3 * resid_dv)

    return serie[(resid < lower) | (resid > upper)]


for area in areas:
    
    serie = train[area].copy()

    anomalies = detect_anomalies(serie)

    anomalies
    
    serie.loc[anomalies.index] = np.nan

    serie = serie.interpolate(method="linear")

    train[area] = serie


2270     0.0
2272     0.0
2281     0.0
2366     0.0
2457     0.0
        ... 
6309    11.0
6396     4.0
6398    12.0
6405    12.0
6408    13.0
Name: Pickup Community Area_0, Length: 174, dtype: float64
62      1.0
261     6.0
397     0.0
494     0.0
495     0.0
       ... 
6060    0.0
6067    0.0
6068    3.0
6118    3.0
6193    0.0
Name: Pickup Community Area_1, Length: 128, dtype: float64
19      0.0
45      0.0
46      1.0
342     0.0
538     0.0
       ... 
6285    0.0
6293    0.0
6300    0.0
6309    0.0
6371    1.0
Name: Pickup Community Area_2, Length: 160, dtype: float64
415      1.0
615      1.0
634      0.0
991      8.0
1245    15.0
        ... 
6175     0.0
6271     0.0
6277     0.0
6316     0.0
6374     1.0
Name: Pickup Community Area_3, Length: 163, dtype: float64
602     0.0
632     0.0
1014    0.0
1302    4.0
1369    5.0
       ... 
6194    0.0
6302    0.0
6371    0.0
6396    1.0
6422    0.0
Name: Pickup Community Area_4, Length: 155, dtype: float64
589     0.0
1157    3.0

Apply Log Transformation

save cleaned train time series

In [6]:
import joblib
joblib.dump(train, "cleaned_train.pkl")

['cleaned_train.pkl']

In [7]:
for area in areas:
    train[area] = np.log1p(train[area] + 1)

Check and make time series stationary for each Area with Augmented Dickey fuller test and differencing

In [15]:
from statsmodels.tsa.stattools import adfuller
# import matplotlib.pyplot as plt


def check_stationarity(serie, significance_level=0.05):
    """
    Checks if a time series is stationary using the ADF test.
    Handles NaNs automatically before testing.
    """
    # Drop missing values caused by differencing
    clean_serie = pd.Series(serie).dropna()
    
    # Handle edge case: empty series or zero variance
    if len(clean_serie) < 10 or clean_serie.nunique() <= 1:
        return False

    result = adfuller(clean_serie, autolag='AIC')
    p_value = result[1]

    return p_value < significance_level


def make_stationary_diff(serie, max_diff=3, significance_level=0.05):
    """
    Iteratively differences a time series until it becomes stationary 
    or reaches max_diff. Returns the transformed series and total differences applied.
    """
    current_serie = pd.Series(serie).copy()
    diff_count = 0

    while diff_count < max_diff:
        if check_stationarity(current_serie, significance_level):
            break
            
        current_serie = current_serie.diff().dropna()
        diff_count += 1

    return current_serie, diff_count

# serie = None

try:
    for area in areas:
        serie, diff_count = make_stationary_diff(train[area])
        train[area] = serie
except Exception as e:
    print(f"error : {str(e)}")


Fit Naive averages using Global average for each Area